# 01 — Fine-Tuning: Vehicle Re-ID + Text-Vehicle Joint Embedding
**Project:** Text-to-Vehicle Retrieval & Tracking from CCTV footage

**Requires attached Kaggle datasets:** VeRi-776, CityFlow-NL (private, access-approved).

## Part A — Setup

In [1]:
%%capture
!pip install -q timm
!pip install -q open_clip_torch
!pip install -q scikit-learn
!pip install -q opencv-python-headless
print("Setup complete.")

In [2]:
import os, glob, re, json, random
from collections import defaultdict

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms
from PIL import Image
import timm
import open_clip

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

PROJECT_ROOT = "/kaggle/working/vehicle_nl_retrieval"
CHECKPOINTS_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

Device: cuda


In [3]:
def save_checkpoint_keep_latest(obj, save_dir, prefix, epoch, keep=2):
    #this saves the latest 2 models and deletes the previous one
    new_path = os.path.join(save_dir, f"{prefix}_epoch{epoch}.pt")
    torch.save(obj, new_path)
    existing = sorted(
        glob.glob(os.path.join(save_dir, f"{prefix}_epoch*.pt")),
        key=lambda p: int(p.split("epoch")[-1].split(".")[0]),
    )
    for old_path in existing[:-keep]:
        os.remove(old_path)
    return new_path


def find_latest_checkpoint(save_dir, prefix):
    candidates = sorted(
        glob.glob(os.path.join(save_dir, f"{prefix}_epoch*.pt")),
        key=lambda p: int(p.split("epoch")[-1].split(".")[0]),
    )
    return candidates[-1] if candidates else None

## Part B — Re-ID pretraining on VeRi-776

In [4]:
import xml.etree.ElementTree as ET

def load_veri_xml_labels(xml_path):
    """Parses train_label.xml / test_label.xml -> dict: filename -> (vehicleID, cameraID, colorID, typeID).

    VeRi-776's XML files declare encoding="gb2312" in their prolog, which
    Python's built-in expat parser can't handle directly (multi-byte encodings
    aren't supported at the file-parsing level). Decoding to a string first
    and parsing from that string sidesteps the issue entirely.
    """
    with open(xml_path, "rb") as f:
        raw_bytes = f.read()

    # Decode using gb2312 (falls back to gbk, which is a superset, if gb2312 fails)
    try:
        xml_text = raw_bytes.decode("gb2312")
    except UnicodeDecodeError:
        xml_text = raw_bytes.decode("gbk", errors="replace")

    # Strip/replace the encoding declaration in the prolog -- now irrelevant
    # since we're parsing from an already-decoded Python string
    xml_text = xml_text.replace('encoding="gb2312"', 'encoding="utf-8"')
    xml_text = xml_text.replace("encoding='gb2312'", "encoding='utf-8'")

    root = ET.fromstring(xml_text)
    items_node = root.find("Items")

    labels = {}
    for item in items_node.findall("Item"):
        labels[item.get("imageName")] = {
            "vehicle_id": int(item.get("vehicleID")),
            "camera_id": int(item.get("cameraID").lstrip("c")),
            "color_id": int(item.get("colorID")),
            "type_id": int(item.get("typeID")),
        }
    return labels

def load_attribute_names(list_path):
    """Parses list_color.txt / list_type.txt -> dict: id -> name"""
    names = {}
    with open(list_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                names[int(parts[0])] = parts[1]
    return names


veri_candidates = glob.glob("/kaggle/input/datasets/*/*veri*") + glob.glob("/kaggle/input/datasets/*/*VeRi*")
assert veri_candidates, "No VeRi-776 dataset found under /kaggle/input/ -- attach it via Add Input."
VERI_ROOT = veri_candidates[0]

def find_subdir(root, name):
    matches = glob.glob(os.path.join(root, "**", name), recursive=True)
    return matches[0] if matches else None

def find_file(root, name):
    matches = glob.glob(os.path.join(root, "**", name), recursive=True)
    return matches[0] if matches else None

TRAIN_DIR = find_subdir(VERI_ROOT, "image_train")
QUERY_DIR = find_subdir(VERI_ROOT, "image_query")
GALLERY_DIR = find_subdir(VERI_ROOT, "image_test")
assert TRAIN_DIR and QUERY_DIR and GALLERY_DIR, f"Could not find image_train/query/test under {VERI_ROOT}"

TRAIN_XML = find_file(VERI_ROOT, "train_label.xml")
TEST_XML = find_file(VERI_ROOT, "test_label.xml")
COLOR_LIST = find_file(VERI_ROOT, "list_color.txt")
TYPE_LIST = find_file(VERI_ROOT, "list_type.txt")
GT_FILE = find_file(VERI_ROOT, "gt_index.txt")
JK_FILE = find_file(VERI_ROOT, "jk_index.txt")
NAME_QUERY_FILE = find_file(VERI_ROOT, "name_query.txt")
NAME_TEST_FILE = find_file(VERI_ROOT, "name_test.txt")
TEST_TRACK_FILE = find_file(VERI_ROOT, "test_track.txt")
assert TRAIN_XML and TEST_XML, "Could not find train_label.xml / test_label.xml"

train_labels = load_veri_xml_labels(TRAIN_XML)
test_labels = load_veri_xml_labels(TEST_XML)
color_names = load_attribute_names(COLOR_LIST)
type_names = load_attribute_names(TYPE_LIST)
print(f"Color classes: {len(color_names)} -> {color_names}")
print(f"Type classes: {len(type_names)} -> {type_names}")

# Build training item list directly from the XML (authoritative) rather than filename regex
train_items = []
for fname, meta in train_labels.items():
    path = os.path.join(TRAIN_DIR, fname)
    if os.path.exists(path):
        train_items.append((path, meta["vehicle_id"], meta["camera_id"], meta["color_id"], meta["type_id"]))

unique_pids = sorted(set(i[1] for i in train_items))
pid2label = {pid: i for i, pid in enumerate(unique_pids)}
NUM_CLASSES = len(unique_pids)
NUM_COLORS = max(color_names.keys())
NUM_TYPES = max(type_names.keys())
print(f"Training identities: {NUM_CLASSES}, images: {len(train_items)}")
print(f"Color classes: {NUM_COLORS}, type classes: {NUM_TYPES}")

Color classes: 10 -> {1: 'yellow', 2: 'orange', 3: 'green', 4: 'gray', 5: 'red', 6: 'blue', 7: 'white', 8: 'golden', 9: 'brown', 10: 'black'}
Type classes: 9 -> {1: 'sedan', 2: 'suv', 3: 'van', 4: 'hatchback', 5: 'mpv', 6: 'pickup', 7: 'bus', 8: 'truck', 9: 'estate'}
Training identities: 575, images: 37746
Color classes: 10, type classes: 9


In [5]:
class ReIDModel(nn.Module):
    def __init__(self, num_classes, num_colors=None, num_types=None, embed_dim=2048):
        super().__init__()
        self.backbone = timm.create_model("resnet50", pretrained=True, num_classes=0, global_pool="avg")
        self.bnneck = nn.BatchNorm1d(embed_dim)
        self.bnneck.bias.requires_grad_(False)
        self.classifier = nn.Linear(embed_dim, num_classes, bias=False)
        self.color_classifier = nn.Linear(embed_dim, num_colors, bias=False) if num_colors else None
        self.type_classifier = nn.Linear(embed_dim, num_types, bias=False) if num_types else None

    def forward(self, x):
        feat = self.backbone(x)
        bn_feat = self.bnneck(feat)
        logits = self.classifier(bn_feat)
        color_logits = self.color_classifier(bn_feat) if self.color_classifier else None
        type_logits = self.type_classifier(bn_feat) if self.type_classifier else None
        return feat, bn_feat, logits, color_logits, type_logits


class VeRiDataset(Dataset):
    def __init__(self, items, transform, pid2label=None):
        self.items = items  
        self.transform = transform
        self.pid2label = pid2label

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, pid, camid, color_id, type_id = self.items[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        label = self.pid2label[pid] if self.pid2label is not None else pid
        return img, label, camid, color_id - 1, type_id - 1


class PKSampler(Sampler):
    """Yields batches of P identities x K images each, for triplet loss."""
    def __init__(self, items, pid2label, P=8, K=4):
        self.P, self.K = P, K
        self.pid_to_indices = {}
        for idx, item in enumerate(items):
            pid = item[1]
            self.pid_to_indices.setdefault(pid2label[pid], []).append(idx)
        self.labels = list(self.pid_to_indices.keys())
        self.length = len(items) // (P * K)

    def __len__(self):
        return self.length

    def __iter__(self):
        for _ in range(self.length):
            batch = []
            chosen_labels = random.sample(self.labels, self.P)
            for lbl in chosen_labels:
                idxs = self.pid_to_indices[lbl]
                replace = len(idxs) < self.K
                batch.extend(np.random.choice(idxs, self.K, replace=replace).tolist())
            yield batch


def batch_hard_triplet_loss(features, labels, margin=0.3):
    dist_mat = torch.cdist(features, features, p=2)
    labels = labels.unsqueeze(1)
    same_id = labels.eq(labels.t())
    dist_ap, dist_an = [], []
    for i in range(dist_mat.size(0)):
        pos_mask = same_id[i].clone()
        pos_mask[i] = False
        neg_mask = ~same_id[i]
        dist_ap.append(dist_mat[i][pos_mask].max())
        dist_an.append(dist_mat[i][neg_mask].min())
    dist_ap = torch.stack(dist_ap)
    dist_an = torch.stack(dist_an)
    return F.relu(dist_ap - dist_an + margin).mean()


print("Re-ID classes and losses defined.")

Re-ID classes and losses defined.


In [ ]:
IMG_SIZE = (256, 256)
P, K = 8, 4  # identities x images per identity per batch
REID_NUM_EPOCHS = 20   # total epochs target -- adjust based on GPU quota
REID_LR = 3.5e-4
sampler = PKSampler(train_items, pid2label, P=P, K=K)

train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

reid_model = ReIDModel(num_classes=NUM_CLASSES, num_colors=NUM_COLORS, num_types=NUM_TYPES).to(device)

start_epoch = 0
latest_reid_ckpt = find_latest_checkpoint(CHECKPOINTS_DIR, "reid")
if latest_reid_ckpt:
    print(f"Resuming from {latest_reid_ckpt}")
    reid_model.load_state_dict(torch.load(latest_reid_ckpt, map_location=device))
    start_epoch = int(latest_reid_ckpt.split("epoch")[-1].split(".")[0])

id_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
color_criterion = nn.CrossEntropyLoss()
type_criterion = nn.CrossEntropyLoss()
COLOR_LOSS_WEIGHT = 0.5
TYPE_LOSS_WEIGHT = 0.5

reid_optimizer = torch.optim.Adam(reid_model.parameters(), lr=REID_LR, weight_decay=5e-4)
reid_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(reid_optimizer, T_max=REID_NUM_EPOCHS)
for _ in range(start_epoch):
    reid_scheduler.step()
reid_scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

train_dataset = VeRiDataset(train_items, train_transform, pid2label)
train_loader = DataLoader(train_dataset, batch_sampler=sampler, num_workers=0)

reid_model.train()
for epoch in range(start_epoch, REID_NUM_EPOCHS):
    total_loss, total_id, total_tri, total_color, total_type, n_batches = 0.0, 0.0, 0.0, 0.0, 0.0, 0
    for imgs, labels, _, color_ids, type_ids in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        color_ids, type_ids = color_ids.to(device), type_ids.to(device)
        reid_optimizer.zero_grad()

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            feat, bn_feat, logits, color_logits, type_logits = reid_model(imgs)
            id_loss = id_criterion(logits, labels)
            tri_loss = batch_hard_triplet_loss(feat.float(), labels, margin=0.3)
            color_loss = color_criterion(color_logits, color_ids)
            type_loss = type_criterion(type_logits, type_ids)
            loss = id_loss + tri_loss + COLOR_LOSS_WEIGHT * color_loss + TYPE_LOSS_WEIGHT * type_loss

        if reid_scaler:
            reid_scaler.scale(loss).backward(); reid_scaler.step(reid_optimizer); reid_scaler.update()
        else:
            loss.backward(); reid_optimizer.step()

        total_loss += loss.item(); total_id += id_loss.item(); total_tri += tri_loss.item()
        total_color += color_loss.item(); total_type += type_loss.item(); n_batches += 1

    reid_scheduler.step()
    print(f"[Re-ID] Epoch {epoch+1}/{REID_NUM_EPOCHS} | loss={total_loss/n_batches:.4f} | "
          f"id={total_id/n_batches:.4f} | triplet={total_tri/n_batches:.4f} | "
          f"color={total_color/n_batches:.4f} | type={total_type/n_batches:.4f} | "
          f"lr={reid_scheduler.get_last_lr()[0]:.6f}")

    save_checkpoint_keep_latest(reid_model.state_dict(), CHECKPOINTS_DIR, "reid", epoch + 1)

print("Re-ID pretraining complete.")

In [8]:
IMG_SIZE = (256, 256)

def load_name_list(path):
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]

def load_relevance_file(path, resolve_to_filenames=None):
    """gt_index.txt / jk_index.txt: one line per query, space-separated tokens.
    Tokens are either filenames directly, or 1-indexed line numbers into
    name_test.txt -- auto-detected from the first token."""
    with open(path) as f:
        lines = [line.strip().split() for line in f]

    if not lines or not lines[0]:
        return lines

    first_token = lines[0][0]
    is_numeric_index = first_token.isdigit()

    if is_numeric_index and resolve_to_filenames is not None:
        resolved = []
        for tokens in lines:
            # try both 1-indexed and 0-indexed in case the mirror differs
            resolved.append([resolve_to_filenames[int(t) - 1] for t in tokens])
        return resolved
    return lines


def load_test_tracks(path):
    """test_track.txt: one line per track, space-separated filenames (~6 images/track, one camera)."""
    tracks = []
    with open(path) as f:
        for line in f:
            fnames = [t for t in line.strip().split() if t.endswith(".jpg")]
            if fnames:
                tracks.append(fnames)
    return tracks


query_names = load_name_list(NAME_QUERY_FILE)
gallery_names = load_name_list(NAME_TEST_FILE)
gt_lists = load_relevance_file(GT_FILE, resolve_to_filenames=gallery_names)
jk_lists = load_relevance_file(JK_FILE, resolve_to_filenames=gallery_names)  
test_tracks = load_test_tracks(TEST_TRACK_FILE)


filename_to_track_idx = {}
for t_idx, fnames in enumerate(test_tracks):
    for fn in fnames:
        filename_to_track_idx[fn] = t_idx

img_transform_eval = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def embed_image(path):
    img = Image.open(path).convert("RGB")
    img = img_transform_eval(img).unsqueeze(0).to(device)
    reid_model.eval()
    _, bn_feat, _, _, _ = reid_model(img)
    return F.normalize(bn_feat[0], dim=-1)

@torch.no_grad()
def embed_track(fnames, image_dir):
    imgs = torch.stack([img_transform_eval(Image.open(os.path.join(image_dir, f)).convert("RGB"))
                         for f in fnames]).to(device)
    reid_model.eval()
    _, bn_feat, _, _, _ = reid_model(imgs)
    return F.normalize(bn_feat.mean(dim=0), dim=-1)

gallery_embeds, gallery_filenames = [], []
processed_tracks = set()
for fname in gallery_names:
    t_idx = filename_to_track_idx.get(fname)
    if t_idx is not None:
        if t_idx in processed_tracks:
            continue  # already embedded this track via an earlier filename
        emb = embed_track(test_tracks[t_idx], GALLERY_DIR)
        for f in test_tracks[t_idx]:
            gallery_filenames.append(f)
            gallery_embeds.append(emb)
        processed_tracks.add(t_idx)
    else:
        gallery_embeds.append(embed_image(os.path.join(GALLERY_DIR, fname)))
        gallery_filenames.append(fname)

gallery_embeds = torch.stack(gallery_embeds)
gallery_index = {f: i for i, f in enumerate(gallery_filenames)}


all_ap, all_cmc = [], []
for q_idx, q_name in enumerate(query_names):
    q_embed = embed_image(os.path.join(QUERY_DIR, q_name))
    sims = (gallery_embeds @ q_embed).cpu().numpy()
    order = np.argsort(-sims)

    gt_set = set(gt_lists[q_idx])
    jk_set = set(jk_lists[q_idx])

    ranked_filenames = [gallery_filenames[i] for i in order]
    filtered = [f for f in ranked_filenames if f not in jk_set] 
    matches = np.array([1 if f in gt_set else 0 for f in filtered])

    if matches.sum() == 0:
        continue

    cmc = matches.cumsum()
    cmc[cmc > 1] = 1
    all_cmc.append(cmc[:10])

    precision_at_k = matches.cumsum() / (np.arange(len(matches)) + 1)
    all_ap.append((precision_at_k * matches).sum() / matches.sum())

all_cmc = np.array(all_cmc).mean(axis=0)
print(f"mAP: {np.mean(all_ap):.4f}")
print(f"Rank-1: {all_cmc[0]:.4f}, Rank-5: {all_cmc[4]:.4f}, Rank-10: {all_cmc[9]:.4f}")

mAP: 0.6872
Rank-1: 0.9100, Rank-5: 0.9160, Rank-10: 0.9541


## Part C — Extract vehicle crops from CityFlow-NL (selective, storage-efficient)

In [6]:
#-------------UNCOMMENT & USE THIS CELL IF YOU HAVE ALREADY HAVE A PRE-TRAINED MODEL ON VeRI dataset-------------------

#reid_ckpt_candidates = [p for p in glob.glob("/kaggle/input/**/*.pt", recursive=True)
#                         if "reid_epoch20" in p.lower()]
#assert reid_ckpt_candidates, "reid_epoch20.pt not found under /kaggle/input/ -- attach that dataset via Add Input."
#REID_CKPT_PATH = reid_ckpt_candidates[0]
#print("Using pre-trained Re-ID checkpoint:", REID_CKPT_PATH)

#raw_state_dict = torch.load(REID_CKPT_PATH, map_location=device)

# Infer architecture directly from the checkpoint's own weight shapes --
# works whether this was trained with the old 2-argument ReIDModel or the
# newer version with color/type auxiliary heads, no guessing required.
#num_classes_ckpt = raw_state_dict["classifier.weight"].shape[0]
#embed_dim_ckpt = raw_state_dict["classifier.weight"].shape[1]
#num_colors_ckpt = raw_state_dict["color_classifier.weight"].shape[0] if "color_classifier.weight" in raw_state_dict else None
#num_types_ckpt = raw_state_dict["type_classifier.weight"].shape[0] if "type_classifier.weight" in raw_state_dict else None

#print(f"Inferred from checkpoint -- num_classes: {num_classes_ckpt}, embed_dim: {embed_dim_ckpt}, "
#      f"num_colors: {num_colors_ckpt}, num_types: {num_types_ckpt}")

#reid_model = ReIDModel(
#    num_classes=num_classes_ckpt,
#    num_colors=num_colors_ckpt,
#    num_types=num_types_ckpt,
#    embed_dim=embed_dim_ckpt,
#).to(device)
#reid_model.load_state_dict(raw_state_dict)
#reid_model.eval()

#print("Re-ID checkpoint loaded successfully -- skipping Part B training, proceeding to CityFlow-NL fine-tuning.")

Using pre-trained Re-ID checkpoint: /kaggle/input/datasets/takayanagisensei/reid-epoch13/reid_epoch20.pt
Inferred from checkpoint -- num_classes: 575, embed_dim: 2048, num_colors: 10, num_types: 9


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Re-ID checkpoint loaded successfully -- skipping Part B training, proceeding to CityFlow-NL fine-tuning.


In [9]:
cityflow_candidates = [d for d in glob.glob("/kaggle/input/datasets/*/*") if "cityflow" in d.lower()]
assert cityflow_candidates, "No CityFlow-NL dataset found under /kaggle/input/ -- attach it via Add Input."
CITYFLOW_INPUT_ROOT = cityflow_candidates[0]
source_data_dir = os.path.join(CITYFLOW_INPUT_ROOT, "data")
assert os.path.isdir(source_data_dir), f"No 'data' folder found under {CITYFLOW_INPUT_ROOT}"

CITYFLOW_NL_DIR = os.path.join(PROJECT_ROOT, "data/cityflow_nl")
os.makedirs(os.path.dirname(CITYFLOW_NL_DIR), exist_ok=True)
if os.path.islink(CITYFLOW_NL_DIR):
    os.unlink(CITYFLOW_NL_DIR)
os.symlink(source_data_dir, CITYFLOW_NL_DIR)
print("Linked CityFlow-NL data to:", CITYFLOW_NL_DIR)

with open(os.path.join(CITYFLOW_NL_DIR, "train-tracks.json")) as f:
    train_tracks = json.load(f)
with open(os.path.join(CITYFLOW_NL_DIR, "test-tracks.json")) as f:
    test_tracks = json.load(f)
with open(os.path.join(CITYFLOW_NL_DIR, "test-queries.json")) as f:
    test_queries = json.load(f)
print(f"Train tracks: {len(train_tracks)}, test tracks: {len(test_tracks)}, test queries: {len(test_queries)}")

Linked CityFlow-NL data to: /kaggle/working/vehicle_nl_retrieval/data/cityflow_nl
Train tracks: 2155, test tracks: 184, test queries: 184


In [10]:
CROPS_OUT_ROOT = os.path.join(PROJECT_ROOT, "data/cityflow_nl_crops")
os.makedirs(CROPS_OUT_ROOT, exist_ok=True)

# Build a manifest of exactly which frames are needed per camera video --
# extracting only these (cropped, never the full frame) keeps storage tiny
# compared to dumping every frame of every camera.
needed_crops = defaultdict(lambda: defaultdict(list))

def collect_crops(tracks_dict):
    for uuid, track in tracks_dict.items():
        for frel, box in zip(track["frames"], track["boxes"]):
            frel_norm = frel.lstrip("./")
            video_dir = os.path.dirname(os.path.dirname(frel_norm))
            frame_num = int(os.path.splitext(os.path.basename(frel_norm))[0])
            needed_crops[video_dir][frame_num].append((uuid, box))

collect_crops(train_tracks)
collect_crops(test_tracks)
print(f"Cameras needing extraction: {len(needed_crops)}")
print(f"Total (video, frame) pairs: {sum(len(v) for v in needed_crops.values())}")

Cameras needing extraction: 51
Total (video, frame) pairs: 81829


In [11]:
for video_dir_rel, frame_map in needed_crops.items():
    vdo_path = os.path.join(CITYFLOW_NL_DIR, video_dir_rel, "vdo.avi")
    if not os.path.exists(vdo_path):
        print(f"Missing video, skipping: {vdo_path}")
        continue

    target_frames = set(frame_map.keys())
    cap = cv2.VideoCapture(vdo_path)
    frame_idx, saved = 0, 0
    max_needed = max(target_frames)

    while target_frames:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1  # CityFlow frame numbering is 1-indexed
        if frame_idx in target_frames:
            for uuid, (x, y, w, h) in frame_map[frame_idx]:
                out_dir = os.path.join(CROPS_OUT_ROOT, uuid)
                out_path = os.path.join(out_dir, f"{frame_idx:06d}.jpg")
                if os.path.exists(out_path):
                    continue  # already extracted -- resumable across reruns
                crop = frame[y:y+h, x:x+w]
                if crop.size == 0:
                    continue
                os.makedirs(out_dir, exist_ok=True)
                cv2.imwrite(out_path, crop, [cv2.IMWRITE_JPEG_QUALITY, 90])
                saved += 1
            target_frames.discard(frame_idx)
        if frame_idx >= max_needed and not target_frames:
            break
    cap.release()
    print(f"{video_dir_rel}: saved {saved} new crops")

print("Crop extraction complete.")

validation/S02/c006: saved 5075 new crops
validation/S02/c007: saved 4197 new crops
validation/S02/c009: saved 8227 new crops
validation/S02/c008: saved 3345 new crops
validation/S05/c016: saved 10845 new crops
validation/S05/c010: saved 4151 new crops
validation/S05/c017: saved 5981 new crops
validation/S05/c018: saved 2486 new crops
validation/S05/c019: saved 611 new crops
validation/S05/c020: saved 2348 new crops
validation/S05/c021: saved 5878 new crops
validation/S05/c022: saved 12478 new crops
validation/S05/c023: saved 4133 new crops
validation/S05/c024: saved 824 new crops
validation/S05/c025: saved 5736 new crops
validation/S05/c026: saved 13118 new crops
validation/S05/c027: saved 11141 new crops
validation/S05/c028: saved 8236 new crops
validation/S05/c029: saved 12694 new crops
validation/S05/c033: saved 23865 new crops
validation/S05/c034: saved 14963 new crops
validation/S05/c035: saved 13626 new crops
validation/S05/c036: saved 11241 new crops
train/S01/c003: saved 2410 

## Part D — Contrastive text-vehicle training on CityFlow-NL

In [15]:
SHARED_DIM = 256

class VisualEncoder(nn.Module):
    def __init__(self, reid_model, in_dim=2048, out_dim=SHARED_DIM, freeze_backbone=False):
        super().__init__()
        self.reid_model = reid_model
        if freeze_backbone:
            for p in self.reid_model.parameters():
                p.requires_grad_(False)
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        _, bn_feat, _, _, _ = self.reid_model(x)
        return F.normalize(self.proj(bn_feat), dim=-1)


class TextEncoder(nn.Module):
    def __init__(self, clip_model, in_dim, out_dim=SHARED_DIM, freeze_backbone=True):
        super().__init__()
        self.clip_model = clip_model
        if freeze_backbone:
            for p in self.clip_model.parameters():
                p.requires_grad_(False)
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, tokenized_text):
        text_feat = self.clip_model.encode_text(tokenized_text).float()
        return F.normalize(self.proj(text_feat), dim=-1)


def contrastive_loss(image_embeds, text_embeds, logit_scale):
    scale = logit_scale.exp()
    logits_per_image = scale * image_embeds @ text_embeds.t()
    logits_per_text = logits_per_image.t()
    targets = torch.arange(image_embeds.size(0), device=image_embeds.device)
    return (F.cross_entropy(logits_per_image, targets) + F.cross_entropy(logits_per_text, targets)) / 2


class CityFlowNLTrackDataset(Dataset):
    def __init__(self, tracks_dict, crops_root, transform):
        self.tracks = list(tracks_dict.items())
        self.crops_root = crops_root
        self.transform = transform

    def __len__(self):
        return len(self.tracks)

    def __getitem__(self, idx):
        uuid, track = self.tracks[idx]
        frame_idx = random.randrange(len(track["frames"]))
        frame_num = int(os.path.splitext(os.path.basename(track["frames"][frame_idx]))[0])
        crop_path = os.path.join(self.crops_root, uuid, f"{frame_num:06d}.jpg")
        img = Image.open(crop_path).convert("RGB")
        img = self.transform(img)
        nl_desc = random.choice(track["nl"])
        return img, nl_desc, uuid


print("Joint model classes defined.")

Joint model classes defined.


In [17]:
clip_model, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model = clip_model.to(device)
CLIP_TEXT_DIM = clip_model.text_projection.shape[1] if hasattr(clip_model, "text_projection") else 512

def collate_fn(batch):
    imgs, texts, uuids = zip(*batch)
    imgs = torch.stack(imgs)
    tokens = tokenizer(list(texts))
    return imgs, tokens, uuids

img_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

joint_train_dataset = CityFlowNLTrackDataset(train_tracks, CROPS_OUT_ROOT, img_transform)
joint_train_loader = DataLoader(joint_train_dataset, batch_size=32, shuffle=True,
                                 num_workers=0, collate_fn=collate_fn)
print("Joint train batches/epoch:", len(joint_train_loader))

# Load the fully-trained Re-ID backbone as the starting point for the visual encoder

final_reid_ckpt = find_latest_checkpoint(CHECKPOINTS_DIR, "reid")
assert final_reid_ckpt, "No Re-ID checkpoint found -- run Part B first."
reid_model = ReIDModel(num_classes=NUM_CLASSES).to(device)
reid_model.load_state_dict(torch.load(final_reid_ckpt, map_location=device))

visual_encoder = VisualEncoder(reid_model, freeze_backbone=False).to(device)
text_encoder = TextEncoder(clip_model, in_dim=CLIP_TEXT_DIM, freeze_backbone=True).to(device)
logit_scale = nn.Parameter(torch.ones([], device=device) * np.log(1 / 0.07))

Joint train batches/epoch: 68


In [18]:
JOINT_NUM_EPOCHS = 20
JOINT_LR = 1e-4

joint_start_epoch = 0
latest_joint_ckpt = find_latest_checkpoint(CHECKPOINTS_DIR, "joint_model")
if latest_joint_ckpt:
    print(f"Resuming joint training from {latest_joint_ckpt}")
    ck = torch.load(latest_joint_ckpt, map_location=device)
    visual_encoder.load_state_dict(ck["visual_encoder"])
    text_encoder.proj.load_state_dict(ck["text_encoder_proj"])
    logit_scale = ck["logit_scale"]
    joint_start_epoch = int(latest_joint_ckpt.split("epoch")[-1].split(".")[0])

params = list(visual_encoder.parameters()) + list(text_encoder.proj.parameters()) + [logit_scale]
joint_optimizer = torch.optim.Adam(params, lr=JOINT_LR, weight_decay=1e-4)
joint_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(joint_optimizer, T_max=JOINT_NUM_EPOCHS)
for _ in range(joint_start_epoch):
    joint_scheduler.step()
joint_scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

visual_encoder.train()
text_encoder.train()

for epoch in range(joint_start_epoch, JOINT_NUM_EPOCHS):
    total_loss, n_batches = 0.0, 0
    for imgs, tokens, _ in joint_train_loader:
        imgs, tokens = imgs.to(device), tokens.to(device)
        joint_optimizer.zero_grad()
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            image_embeds = visual_encoder(imgs)
            text_embeds = text_encoder(tokens)
            loss = contrastive_loss(image_embeds, text_embeds, logit_scale)
        if joint_scaler:
            joint_scaler.scale(loss).backward()
            joint_scaler.step(joint_optimizer)
            joint_scaler.update()
        else:
            loss.backward()
            joint_optimizer.step()
        total_loss += loss.item(); n_batches += 1

    joint_scheduler.step()
    print(f"[Joint] Epoch {epoch+1}/{JOINT_NUM_EPOCHS} | loss={total_loss/n_batches:.4f} | "
          f"lr={joint_scheduler.get_last_lr()[0]:.6f}")

    save_checkpoint_keep_latest({
        "visual_encoder": visual_encoder.state_dict(),
        "text_encoder_proj": text_encoder.proj.state_dict(),
        "logit_scale": logit_scale,
    }, CHECKPOINTS_DIR, "joint_model", epoch + 1)

print("Joint training complete.")

[Joint] Epoch 1/20 | loss=2.7473 | lr=0.000099
[Joint] Epoch 2/20 | loss=2.1753 | lr=0.000098
[Joint] Epoch 3/20 | loss=1.9834 | lr=0.000095
[Joint] Epoch 4/20 | loss=1.8595 | lr=0.000090
[Joint] Epoch 5/20 | loss=1.7415 | lr=0.000085
[Joint] Epoch 6/20 | loss=1.6576 | lr=0.000079
[Joint] Epoch 7/20 | loss=1.5925 | lr=0.000073
[Joint] Epoch 8/20 | loss=1.4737 | lr=0.000065
[Joint] Epoch 9/20 | loss=1.4463 | lr=0.000058
[Joint] Epoch 10/20 | loss=1.3655 | lr=0.000050
[Joint] Epoch 11/20 | loss=1.3073 | lr=0.000042
[Joint] Epoch 12/20 | loss=1.2601 | lr=0.000035
[Joint] Epoch 13/20 | loss=1.2160 | lr=0.000027
[Joint] Epoch 14/20 | loss=1.2115 | lr=0.000021
[Joint] Epoch 15/20 | loss=1.1296 | lr=0.000015
[Joint] Epoch 16/20 | loss=1.1265 | lr=0.000010
[Joint] Epoch 17/20 | loss=1.0789 | lr=0.000005
[Joint] Epoch 18/20 | loss=1.0467 | lr=0.000002
[Joint] Epoch 19/20 | loss=1.0838 | lr=0.000001
[Joint] Epoch 20/20 | loss=1.0472 | lr=0.000000
Joint training complete.


In [19]:
from IPython.display import FileLink
!zip -r output.zip /kaggle/working/vehicle_nl_retrieval/checkpoints
FileLink(r'output.zip')

Final joint checkpoint: /kaggle/working/vehicle_nl_retrieval/checkpoints/joint_model_epoch20.pt


/kaggle/working/vehicle_nl_retrieval/checkpoints/joint_model_epoch20.pt